In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
N   0.000   0.000   1.413
C   0.000   0.000  -1.379
C   0.000   1.139   0.719
C   0.000  -1.139   0.719
C   0.000   1.193  -0.670
C   0.000  -1.193  -0.670
H   0.000   0.000  -2.460
H   0.000   2.053   1.302
H   0.000  -2.053   1.302
H   0.000   2.147  -1.177
H   0.000  -2.147  -1.177'''
molecule = 'C5H5N'
spin = 0
charge =  0


In [3]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [5]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./C5H5N.txt
orbital  energy   occ
0     -15.539583769290934    2.0
1     -11.225946034395871    2.0
2     -11.225843903093566    2.0
3     -11.20816190256241    2.0
4     -11.198089635860331    2.0
5     -11.198038359920178    2.0
6     -1.2089383675807757    2.0
7     -1.0375229979444425    2.0
8     -0.9823175911998596    2.0
9     -0.8070097459552772    2.0
10     -0.800418340473781    2.0
11     -0.6796467842075087    2.0
12     -0.6113695670107573    2.0
13     -0.5958041074620224    2.0
14     -0.5530179253612666    2.0
15     -0.5227621573157746    2.0
16     -0.49797405183057025    2.0
17     -0.46321578743787595    2.0
18     -0.33843629940522857    2.0
19     -0.32640674029003663    2.0
20     -0.3040263917884674    2.0
21     0.23847798795676778    0.0
22     0.24351035258303044    0.0
23     0.4764988061708164    0.0
24     0.5745797042504671    0.0
25     0.6247536291595472    0.0
26     0.6684882170406967    0.0
27     0.7055906258701424    0.0
28     0.70716

In [6]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([17,20,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-246.13838649063197),
 np.float64(-6.7696500218528115),
 FCIvector([[-9.15112058e-01,  0.00000000e+00,  9.90102404e-03,
              3.61228568e-03, -2.53882561e-03,  0.00000000e+00,
              0.00000000e+00, -8.27833686e-02, -7.65609308e-03,
              0.00000000e+00,  0.00000000e+00, -9.28072569e-04,
              1.43156525e-03,  0.00000000e+00,  0.00000000e+00,
             -2.14503138e-02, -1.28215669e-03,  2.22603353e-02,
              0.00000000e+00,  6.79753442e-04],
            [ 0.00000000e+00,  2.57695519e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -1.13846611e-01,
             -1.25011922e-02,  0.00000000e+00,  0.00000000e+00,
             -9.53913924e-03,  4.46764258e-04,  0.00000000e+00,
              0.00000000e+00,  9.42553850e-03,  7.84494314e-04,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
             -2.91123827e-04,  0.00000000e+00],
            [ 9.90102404e-03,  0.00000000e+00,  1.75061792e-01,
    

In [7]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      -0.915112058201282
[0 1 2]      [0 2 3]      0.009901024037651
[0 1 2]      [1 2 3]      0.003612285680089
[0 1 2]      [0 1 4]      -0.002538825605833
[0 1 2]      [0 3 4]      -0.082783368557701
[0 1 2]      [1 3 4]      -0.007656093081804
[0 1 2]      [0 2 5]      -0.000928072568884
[0 1 2]      [1 2 5]      0.001431565250452
[0 1 2]      [2 3 5]      -0.021450313753884
[0 1 2]      [0 4 5]      -0.001282156687784
[0 1 2]      [1 4 5]      0.022260335292859
[0 1 2]      [3 4 5]      0.000679753442081
[0 1 3]      [0 1 3]      0.025769551864307
[0 1 3]      [0 2 4]      -0.113846611482969
[0 1 3]      [1 2 4]      -0.012501192199914
[0 1 3]      [2 3 4]      -0.009539139239864
[0 1 3]      [0 1 5]      0.000446764257770
[0 1 3]      [0 3 5]      0.009425538502360
[0 1 3]      [1 3 5]      0.000784494313543
[0 1 3]      [2 4 5]      -0.000291123827163
[0 2 3]      [0 1 2]      0.009901024037651
[0 2 3]      [0 2 3]      0.175061791603347
[0 2 3]      [1 2 3] 